# REDLINE NASA metadata coverage check

Bu not defteri mevcut APXS hedef satirlari ile saklanmis HiRISE label sinirlarinin **sayisal metadata tutarliligini** yeniden kontrol eder. Raster indirme, piksel/no-data kontrolu, CRS donusumu, kesin numune/delik pozu veya ROS-Unreal testi yapmaz.

Skill: jupyter-notebook. Kaynaklar ve onceki sinirlar: research/notes/SAHA_VE_BILIM_ISTASYONU.md.


In [1]:
from pathlib import Path
import json
import hashlib
import math
import re

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "research/notes/APXS_HEDEF_ON_KONTROL.json").is_file()), None)
if root is None:
    raise RuntimeError("Run from inside the REDLINE repository")
input_paths = {
    "apxs": root / "research/notes/APXS_HEDEF_ON_KONTROL.json",
    "hirise": root / "research/notes/HIRISE_YELLOWKNIFE_METADATA_KONTROLU.json",
}
inputs_sha256 = {k: hashlib.sha256(p.read_bytes()).hexdigest() for k, p in input_paths.items()}
apxs = json.loads(input_paths["apxs"].read_text(encoding="utf-8"))
hirise = json.loads(input_paths["hirise"].read_text(encoding="utf-8"))
print("Loaded local metadata records; no network calls.")


Loaded local metadata records; no network calls.


## Plan and limits

1. Saklanan HiRISE label SHA-256 degerini kontrol et.
2. Label icindeki sinirlari kayit alanlariyla karsilastir.
3. Dort APXS satirinin sayisal enlem/boylam kutu icerimini yeniden hesapla.
4. Onceki kayitla uyusma ve kalan QA islerini raporla.

Ayni kaydin farkli gosterimlerinin uyusmasi bagimsiz saha dogrulamasi degildir. APXS koordinat anlamlari/enlem turu ile goruntunun gercek footprint ve no-data durumu halen aciktir.


In [2]:
label = hirise["label"]
label_hash_matches = hashlib.sha256(label.encode("utf-8")).hexdigest() == hirise["sha256"]
if not label_hash_matches:
    raise ValueError("Saved label bytes do not match the recorded source hash")

def label_number(key):
    match = re.search(r"^\s*" + re.escape(key) + r"\s*=\s*([-+0-9.eE]+)", label, re.MULTILINE)
    if not match:
        raise ValueError("Missing label field: " + key)
    return float(match.group(1))

keys = ["MINIMUM_LATITUDE", "MAXIMUM_LATITUDE", "WESTERNMOST_LONGITUDE", "EASTERNMOST_LONGITUDE", "MAP_SCALE"]
fields_match = all(math.isclose(label_number(k), float(hirise["fields"][k]), rel_tol=0, abs_tol=1e-12) for k in keys)
if not fields_match:
    raise ValueError("Recorded fields disagree with saved label")
print("Product:", hirise["product_id"], "version", hirise["product_version"])
print("Label hash matches:", label_hash_matches)
print("Label fields match:", fields_match)
print("Grid scale (not a guarantee of object resolution):", hirise["fields"]["MAP_SCALE"], "m/pixel")


Product: PSP_010573_1755_RED version 2.0
Label hash matches: True
Label fields match: True
Grid scale (not a guarantee of object resolution): 0.25 m/pixel


In [3]:
bounds = hirise["fields"]
recorded = {row["target"]: row["numeric_bbox_contains"] for row in hirise["checks"]}
results = []
for row in apxs["selected_targets"]:
    lat, lon = float(row["Latitude"]), float(row["Longitude"])
    inside = (bounds["MINIMUM_LATITUDE"] <= lat <= bounds["MAXIMUM_LATITUDE"] and bounds["WESTERNMOST_LONGITUDE"] <= lon <= bounds["EASTERNMOST_LONGITUDE"])
    result = {"target": row["Target"], "sample_site": row["Sample_site"], "sol": row["Sol"], "preparation": row["Preparation_method"], "numeric_bbox_contains": inside}
    results.append(result)
    if recorded.get(row["Target"]) != inside:
        raise ValueError("Previous numeric check differs: " + row["Target"])
    print(json.dumps(result, ensure_ascii=False))


{"target": "Wernecke_Brushed", "sample_site": "John_Klein", "sol": "169", "preparation": "RB", "numeric_bbox_contains": true}
{"target": "CumberlandNew_RP", "sample_site": "Cumberland", "sol": "276", "preparation": "RU", "numeric_bbox_contains": true}
{"target": "Gobabeb_DumpA_APXS", "sample_site": "Gobabeb", "sol": "1226", "preparation": "Postsieve", "numeric_bbox_contains": false}
{"target": "Gobabeb_DumpC_corrected", "sample_site": "Gobabeb", "sol": "1242", "preparation": "Sieve150µm-1mm", "numeric_bbox_contains": false}


In [4]:
summary = {
    "scope": "local metadata consistency; numeric bounding-box containment only",
    "input_sha256": inputs_sha256,
    "saved_label_hash_matches": label_hash_matches,
    "saved_fields_match_label": fields_match,
    "target_rows": len(results),
    "numeric_inside": sum(row["numeric_bbox_contains"] for row in results),
    "results": results,
    "raw_apxs_csv_revalidated": False,
    "coordinate_conventions_reconciled": False,
    "raster_footprint_and_nodata_checked": False,
    "exact_drill_pose_validated": False,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "scope": "local metadata consistency; numeric bounding-box containment only",
  "input_sha256": {
    "apxs": "6f791602fb875932eda1df25002aa023031cc5d128e91acfc238d8708fb99f92",
    "hirise": "693b81c7fe94f10873b9ec3ac05860ddb09bc6f57e08db2b695107eb217db878"
  },
  "saved_label_hash_matches": true,
  "saved_fields_match_label": true,
  "target_rows": 4,
  "numeric_inside": 2,
  "results": [
    {
      "target": "Wernecke_Brushed",
      "sample_site": "John_Klein",
      "sol": "169",
      "preparation": "RB",
      "numeric_bbox_contains": true
    },
    {
      "target": "CumberlandNew_RP",
      "sample_site": "Cumberland",
      "sol": "276",
      "preparation": "RU",
      "numeric_bbox_contains": true
    },
    {
      "target": "Gobabeb_DumpA_APXS",
      "sample_site": "Gobabeb",
      "sol": "1226",
      "preparation": "Postsieve",
      "numeric_bbox_contains": false
    },
    {
      "target": "Gobabeb_DumpC_corrected",
      "sample_site": "Gobabeb",
      "sol":

## Interpretation and next step

Iki Yellowknife hedef satiri bu sayisal kutuda kalir; secili iki Gobabeb satiri kalmaz. Bu sonuc kesin ortusme/konum onayi degildir. Sonraki adim APXS koordinat ve numune-koken semantigini dogrulamak, uygun raster kirpimini edinmek ve gercek footprint/no-data/DEM hizalamasini kontrol etmektir.

Ham APXS CSV bu not defterinde yeniden indirilmedi; kaydedilmis satirlar kullanildi. Yeni kimya, pH, sensor verisi veya 3D arazi uretilmedi.
